<style>
body, .jp-Notebook { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; color: #2c3e50; line-height: 1.6; }
.jp-RenderedHTMLCommon h1 { font-size: 1.9em; color: #1f3a5f; font-weight: 600; border-bottom: 2px solid #e2e8f0; padding-bottom: 0.3em; margin-top: 1.4em; }
.jp-RenderedHTMLCommon h2 { color: #1f3a5f; font-weight: 600; margin-top: 1.2em; }
.jp-RenderedHTMLCommon h3 { color: #2d5a8c; font-weight: 600; }
.jp-RenderedHTMLCommon a { color: #2b6cb0; text-decoration: none; }
.jp-RenderedHTMLCommon a:hover { color: #1a4480; text-decoration: underline; }
.jp-RenderedHTMLCommon code { background-color: #f4f6f8; color: #c7254e; padding: 1px 5px; border-radius: 3px; font-size: 0.9em; }
.jp-RenderedHTMLCommon pre { background-color: #f8f9fb; border: 1px solid #e4e8ed; border-radius: 5px; padding: 10px 12px; }
.jp-CodeMirrorEditor, .CodeMirror, .jp-InputArea-editor { font-size: 0.92em; }
.jp-Cell-inputArea .CodeMirror { background-color: #f8f9fb !important; border-radius: 5px; }
.jp-OutputArea-output pre { background-color: #fcfcfd; }
#title { font-size: 2.4em; font-weight: 700; color: #1f3a5f; margin-bottom: 0.1em; }
#subtitle { font-size: 1.45em; font-weight: 500; color: #2d5a8c; margin-top: 0; margin-bottom: 0.4em; }
#subtitle a { color: #2d5a8c; text-decoration: none; }
#subtitle a::after { content: " \2197"; font-size: 0.6em; font-weight: 700; vertical-align: 0.25em; opacity: 0.75; }
#author { color: #4a5568; font-size: 1em; margin-bottom: 0.1em; }
.affiliation { font-size: 0.85em; color: #718096; }
#date { color: #718096; font-size: 0.9em; margin-bottom: 1.5em; }
#bar { width: 70px; border-bottom: 2px solid #cbd5e0; margin: 0.5em 0 1.5em 0; }
</style>

<div id="title">Program Analysis</div>
<div id="subtitle"><a href="https://hub.crunchdao.com/competitions/broad-obesity-1" target="_blank">Obesity Machine Learning Competition</a></div>
<div id="author">Julie Laffy<br><span class="affiliation">Postdoctoral fellow of the Eric and Wendy Schmidt Center<br>Broad Institute of MIT and Harvard</span></div>
<div id="date">May 2026</div>
<div id="bar"></div>

# Overview

Program analysis methodology used in the Obesity ML Competition.

- **Parts 1–3** — cell-program classification (pre-adipocyte / adipocyte /
  lipogenic / other) and proportions. Methodology used for
  [Crunch 1](https://hub.crunchdao.com/competitions/broad-obesity-1) and
  [Crunch 2](https://hub.crunchdao.com/competitions/broad-obesity-2).
- **Part 4** — thermogenic signature scoring. Methodology used for
  [Crunch 3](https://hub.crunchdao.com/competitions/broad-obesity-3).

# Setup

In [ ]:
# Install required packages
%pip install "pyscalop>=0.2.0" pandas pyarrow --quiet

import numpy as np
import pandas as pd
import pyscalop as ps

# Part 1: All Cells

## 1.1 Preprocess

In [ ]:
full_matrix = "data/m.allgenes.logcpm.parquet"
proc_matrix = "data/m.logcpm.parquet"
gene_cutoff = 3

# read in expression matrix (gene x cell, log2(CPM/10 + 1))
m = pd.read_parquet(full_matrix).set_index("gene")

# calculate average gene expression
avg = ps.aggr_gene_expr(m, is_bulk=False)

# filter genes that do not pass minimum gene expression cutoff
m2 = m.loc[avg >= gene_cutoff]

# save filtered matrix
m2.reset_index().to_parquet(proc_matrix, index=False)

## 1.2 Score Adipo/Preadipo

In [ ]:
proc_matrix = "data/m.logcpm.parquet"
gene_sigs = "sigs/adipo_preadipo_sigs.csv"
frac_genes_retained = 0.5
n_genes_retained = 10
score_results = "data/permuted_scores.adipo_preadipo_sigs.parquet"

# Read in processed expression matrix
m = pd.read_parquet(proc_matrix).set_index("gene")

# Read in gene signatures to score cells by
# there are 2 adipocyte signatures: "adipomap.Ad" and "yi.AD"
# and 2 pre-adipocyte (progenitor) signatures: "adipomap.ASPC" and "yi.ASPC"
sigs = pd.read_csv(gene_sigs).groupby("name")["genes"].agg(list).to_dict()

# Filter out signatures whose genes are poorly represented in {proc_matrix}
# A signature is kept if at least 10 genes and 50% of the signature exist in {proc_matrix}
# {frac_genes_retained} and {n_genes_retained}
sigs_filt = {name: genes for name, genes in sigs.items()
             if sum(g in m.index for g in genes) >= n_genes_retained}

# Calculate single-cell expression scores for each signature and FDRs
scores_permuted = ps.sig_scores(m, sigs_filt, permute=True, conserved_genes=frac_genes_retained)

# Save results
scores_permuted.to_parquet(score_results, index=False)

## 1.3 Classify Adipo/Preadipo/Other

In [ ]:
score_results = "data/permuted_scores.adipo_preadipo_sigs.parquet"
fdr_cutoff = 0.01
diff_cutoff = 0.3

# Load scores
scores = pd.read_parquet(score_results)

# Prepare data: pivot long -> wide, columns are score_<sig> and fdr_<sig>
scores_wide = (scores
    .pivot(index="id", columns="sig", values=["score", "fdr"]))
scores_wide.columns = [f"{val}_{sig}" for val, sig in scores_wide.columns]
scores_wide = scores_wide.reset_index()

scores_wide["max_adipo"] = scores_wide[["score_adipomap.Ad", "score_yi.AD"]].max(axis=1)
scores_wide["max_preadipo"] = scores_wide[["score_adipomap.ASPC", "score_yi.ASPC"]].max(axis=1)
scores_wide["score_diff"] = scores_wide["max_adipo"] - scores_wide["max_preadipo"]
scores_wide["max_overall"] = scores_wide[[
    "score_adipomap.Ad", "score_yi.AD",
    "score_adipomap.ASPC", "score_yi.ASPC",
]].max(axis=1)

# Classify cells
# Adipo logic:
# - score_diff > {diff_cutoff} (adipo score higher than preadipo score)
# - both ASPC (preadipo) signatures must be non-significant (fdr >= {fdr_cutoff})
# - at least one adipocyte signature must be significant ({fdr_cutoff})
# Preadipo logic:
# - score_diff < -{diff_cutoff} (preadipo score higher than adipo score)
# - both adipo signatures must be non-significant (fdr >= {fdr_cutoff})
# - at least one ASPC (preadipocyte) signature must be significant ({fdr_cutoff})
# Other: everything else (ambiguous, dual-identity, neither)

adipo_mask = (
    (scores_wide["score_diff"] > diff_cutoff)
    & (scores_wide["fdr_adipomap.ASPC"] >= fdr_cutoff)
    & (scores_wide["fdr_yi.ASPC"]      >= fdr_cutoff)
    & ((scores_wide["fdr_adipomap.Ad"] < fdr_cutoff) | (scores_wide["fdr_yi.AD"] < fdr_cutoff))
)
preadipo_mask = (
    (scores_wide["score_diff"] < -diff_cutoff)
    & (scores_wide["fdr_adipomap.Ad"] >= fdr_cutoff)
    & (scores_wide["fdr_yi.AD"]      >= fdr_cutoff)
    & ((scores_wide["fdr_adipomap.ASPC"] < fdr_cutoff) | (scores_wide["fdr_yi.ASPC"] < fdr_cutoff))
)
scores_wide["compartment"] = np.where(adipo_mask, "Adipo",
                              np.where(preadipo_mask, "Preadipo", "Other"))

# Extract cell IDs for each compartment
adipo_cells    = scores_wide.loc[scores_wide["compartment"] == "Adipo",    "id"].tolist()
preadipo_cells = scores_wide.loc[scores_wide["compartment"] == "Preadipo", "id"].tolist()
other_cells    = scores_wide.loc[scores_wide["compartment"] == "Other",    "id"].tolist()

# Save cell ID lists (plain text, one cell per line)
def write_cells(cells, path):
    with open(path, "w") as f:
        f.write("
".join(cells))

write_cells(adipo_cells,    "data/cells.adipo.txt")
write_cells(preadipo_cells, "data/cells.preadipo.txt")
write_cells(other_cells,    "data/cells.other.txt")

# Part 2: Adipo Cells

## 2.1 Preprocess

In [ ]:
adipo_cells_path = "data/cells.adipo.txt"
full_matrix = "data/m.allgenes.logcpm.parquet"
proc_matrix = "data/m.adipo.logcpm.parquet"
gene_cutoff = 3

# read expression matrix and cells classified as Adipo
cells = open(adipo_cells_path).read().splitlines()
m = pd.read_parquet(full_matrix).set_index("gene")

# filter to keep only Adipo cells
m_adipo = m[cells]

# calculate average gene expression
avg = ps.aggr_gene_expr(m_adipo, is_bulk=False)

# filter genes that do not pass minimum gene expression cutoff
m2 = m_adipo.loc[avg >= gene_cutoff]

# save filtered matrix
m2.reset_index().to_parquet(proc_matrix, index=False)

## 2.2 Score Lipogenic

In [ ]:
proc_matrix = "data/m.adipo.logcpm.parquet"
gene_sigs = "sigs/lipogenic_sigs.csv"
frac_genes_retained = 0.5
n_genes_retained = 10
score_results = "data/permuted_scores.lipogenic_sigs.parquet"

# Read in processed expression matrix
m = pd.read_parquet(proc_matrix).set_index("gene")

# Read in gene signatures to score cells by
# These are a collection of lipogenic signatures
sigs = pd.read_csv(gene_sigs).groupby("name")["genes"].agg(list).to_dict()

# Filter out signatures whose genes are poorly represented in {proc_matrix}
# A signature is kept if at least 10 genes and 50% of the signature exist in {proc_matrix}
# {frac_genes_retained} and {n_genes_retained}
sigs_filt = {name: genes for name, genes in sigs.items()
             if sum(g in m.index for g in genes) >= n_genes_retained}

# Calculate single-cell expression scores for each signature and FDRs
scores_permuted = ps.sig_scores(m, sigs_filt, permute=True, conserved_genes=frac_genes_retained)

# Save results
scores_permuted.to_parquet(score_results, index=False)

## 2.3 Classify Lipogenic

In [ ]:
score_results = "data/permuted_scores.lipogenic_sigs.parquet"
fdr_cutoff = 0.01
score_cutoff = 0.3
n_sigs = 2

# Load scores
scores = pd.read_parquet(score_results)

# Prepare data: pivot long -> wide
scores_wide = (scores
    .pivot(index="id", columns="sig", values=["score", "fdr"]))
scores_wide.columns = [f"{val}_{sig}" for val, sig in scores_wide.columns]
scores_wide = scores_wide.reset_index()

# Count how many signatures pass criteria per cell
score_cols = [c for c in scores_wide.columns if c.startswith("score_")]
def passes(row):
    n = 0
    for sc in score_cols:
        fdr_c = "fdr_" + sc[len("score_"):]
        if row[sc] > score_cutoff and row[fdr_c] < fdr_cutoff:
            n += 1
    return n
scores_wide["n_pass"] = scores_wide.apply(passes, axis=1)

# Filter for cells that have at least {n_sigs} lipogenic signatures passing cutoffs
lipogenic_cells = scores_wide.loc[scores_wide["n_pass"] >= n_sigs, "id"].tolist()

# Save cell ID list (plain text, one cell per line)
with open("data/cells.lipogenic.txt", "w") as f:
    f.write("
".join(lipogenic_cells))

# Part 3: Proportions

In [ ]:
import os, glob

# get cell assignment file paths
cell_files = sorted(glob.glob("data/cells.*.txt"))

# read in cell assignments
cell_list = {}
for f in cell_files:
    name = os.path.basename(f).replace("cells.", "").replace(".txt", "")
    cell_list[name] = open(f).read().splitlines()

# reorder groups
ord_keys = ["adipo", "preadipo", "other", "lipogenic"]
cell_list = {k: cell_list[k] for k in ord_keys if k in cell_list}

# get cell proportions
n_cells = {k: len(v) for k, v in cell_list.items()}
# lipogenic is a sub-population of adipo; exclude it from the denominator
denom = sum(v for k, v in n_cells.items() if k != "lipogenic")
frac_cells = {k: v / denom for k, v in n_cells.items()}
frac_cells

*Expected output for sample data:*

```
{'adipo':     0.291,
 'preadipo':  0.384,
 'other':     0.325,
 'lipogenic': 0.084}
```

# Part 4: Thermogenic Scoring

The scoring pipeline below is what produced the reference
`*_ThermoScores_cell.csv` and `*_ThermoScores_perturbation.csv` files released
to participants in
[Crunch 3](https://hub.crunchdao.com/competitions/broad-obesity-3). Twelve
signatures are scored, taken from `thermogenic_signatures.csv`:

```
nonucp1.all
REACTOME_AMPK_inhibits_chREBP_R-HSA-163680
REACTOME_Integration_of_energy_metabolism_R-HSA-163685
REACTOME_Mitochondrial_biogenesis_R-HSA-1592230
emont.hAd6
lit.thermogenic
C2.WP_THERMOGENESIS
C5.GOBP_ADAPTIVE_THERMOGENESIS
C5.GOBP_BROWN_FAT_CELL_DIFFERENTIATION
C5.GOBP_NEGATIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS
C5.GOBP_POSITIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS
yi.Thermogenesis_Village
```

## 4.1 Score Thermogenic Signatures

In [ ]:
proc_matrix = "data/m.adipo.logcpm.parquet"
gene_sigs = "sigs/thermogenic_signatures.csv"
n_genes_retained = 2
score_results = "data/permuted_scores.thermo_sigs.parquet"

# Read in processed adipocyte expression matrix
m = pd.read_parquet(proc_matrix).set_index("gene")

# Read in the 12 thermogenic gene signatures
sigs = pd.read_csv(gene_sigs).groupby("name")["genes"].agg(list).to_dict()

# Trim each signature to genes present in the matrix
sigs_present = {name: [g for g in genes if g in m.index] for name, genes in sigs.items()}

# Keep only sigs with at least {n_genes_retained} of their genes in the matrix
sigs_to_score = {name: genes for name, genes in sigs_present.items() if len(genes) >= n_genes_retained}

# Score each signature against each cell (raw score + permutation-based FDR)
scores = ps.sig_scores(m, sigs_to_score, permute=True, conserved_genes=0)

# Annotate each row with how many genes the signature contributed
n_genes_used = {name: len(g) for name, g in sigs_to_score.items()}
scores["n_genes_used"] = scores["sig"].map(n_genes_used)

# Save long per-(cell, sig) table: id, sig, score, fdr, n_genes_used
scores.to_parquet(score_results, index=False)

## 4.2 Z-score and Per-Perturbation Aggregation

Z-score each signature across all adipocyte cells, aggregate to perturbation
level (mean z per gene × sig), and rank perturbations by
`agg_top3_z` (mean of the top-3 signature z-scores). Outputs in the format of
the [released reference files](https://hub.crunchdao.com/competitions/broad-obesity-3)
for Crunch 3: `ThermoScores_cell.csv` (one row per cell) and
`ThermoScores_perturbation.csv` (one row per perturbation, with `agg_top3_z`
and `rank`).

In [ ]:
score_results = "data/permuted_scores.thermo_sigs.parquet"
cell_out = "data/ThermoScores_cell.csv"
perturbation_out = "data/ThermoScores_perturbation.csv"

# Load per-(cell, sig) scores from script 4.1
scores = pd.read_parquet(score_results)

# 1. Z-score each signature across all adipocyte cells
# z = (raw_score - mean_across_cells) / sd_across_cells
scores["z_score"] = scores.groupby("sig")["score"].transform(lambda x: (x - x.mean()) / x.std(ddof=1))

# 2. Derive perturbation (gene) from cell id prefix.
#    TF150 convention only — otherwise use AnnData `.obs['gene']` / cell metadata.
scores["gene"] = scores["id"].str.split("_", n=1).str[0]

# 3. Per-cell wide table: raw score columns + z_-prefixed z-score columns
cell_raw = scores.pivot(index=["id", "gene"], columns="sig", values="score").reset_index()
cell_z   = scores.pivot(index="id",          columns="sig", values="z_score").reset_index()
cell_z.columns = ["id"] + [f"z_{c}" for c in cell_z.columns[1:]]
cell_wide = cell_raw.merge(cell_z, on="id").rename(columns={"id": "cell_id"})
cell_wide.to_csv(cell_out, index=False)

# 4. Per-perturbation aggregation:
#    - mean z-score per (gene, sig) across all cells of that perturbation
#    - agg_top3_z = mean of the top-3 sig z-scores
#    - rank perturbations from highest to lowest agg_top3_z
perturb_long = scores.groupby(["gene", "sig"], as_index=False)["z_score"].mean().rename(columns={"z_score": "mean_z"})
perturb_wide = perturb_long.pivot(index="gene", columns="sig", values="mean_z").reset_index()

sig_cols = [c for c in perturb_wide.columns if c != "gene"]
def top3(row):
    return np.sort(row.values)[::-1][:3].mean()
perturb_wide["agg_top3_z"] = perturb_wide[sig_cols].apply(top3, axis=1)
perturb_wide = perturb_wide.sort_values("agg_top3_z", ascending=False).reset_index(drop=True)
perturb_wide["rank"] = np.arange(1, len(perturb_wide) + 1)

perturb_wide.to_csv(perturbation_out, index=False)

*Below: results from running the pipeline on the **TF150** dataset.*

**`ThermoScores_cell.csv`** — first 3 of 30,975 rows, showing 2 of the
12 raw + 12 z-scored signature columns:

```
                    cell_id  gene  emont.hAd6  z_emont.hAd6
TRIM5_P1_AAACCATTCGCGACAT-1 TRIM5      0.6472        1.9737
CREB1_P1_AAACCCTGTAGTAGGC-1 CREB1      0.1894        0.5775
TRRAP_P1_AAACGAATCAATTGGT-1 TRRAP      0.0647        0.1972
```

**`ThermoScores_perturbation.csv`** — top 8 of 158 perturbations ranked by
`agg_top3_z`, showing the first 2 of the 12 signature columns:

```
    gene  emont.hAd6  lit.thermogenic  agg_top3_z  rank
RNASEH2C      0.5796           0.1700      0.5484     1
 FAM136A      0.1190           0.4887      0.4885     2
 TMEM107     -0.2030           0.3624      0.4199     3
   EP400      0.2438           0.3996      0.3956     4
   TRIM5      0.1003           0.1147      0.3177     5
   CEBPG      0.0505           0.0977      0.2926     6
   MEF2A     -0.0477           0.2802      0.2907     7
    RBAK      0.1003           0.3238      0.2900     8
```